In [7]:
import os
import sys
import shutil  # 🧹 مكتبة إدارة الملفات لحذف الـ Checkpoint
from datetime import datetime, timezone  # <-- أضف timezone هنا
# 1. إعداد مسارات المكتبات لتعمل أوفلاين
offline_packages_path = "/home/jovyan/work/storage/packages"
if offline_packages_path not in sys.path:
    sys.path.insert(0, offline_packages_path)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType

internal_ivy_path = "/home/jovyan/.ivy2/jars/*"

# 2. بناء جلسة سبارك المتزنة مع تفعيل التنظيف الذاتي للـ Checkpoints خلف الكواليس
spark = SparkSession.builder \
    .appName("Kafka_To_Parquet_Incremental_Safe") \
    .config("spark.jars", internal_ivy_path) \
    .config("spark.sql.streaming.minBatchesToRetain", "30") \
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true") \
    .config("spark.sql.streaming.fileSink.log.cleanupDelay", "60000") \
    .getOrCreate()

try:
    print("📡 جاري الاتصال بكافكا وسحب الدفعة التزايدية الجديدة...")
    
    # تحويل القراءة إلى نظام البث المقيد بالدفعة الحالية لضمان تتبع الـ Checkpoint
    kafka_stream_df = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "smarthome-kafka:29092") \
        .option("subscribe", "energy_events") \
        .option("startingOffsets", "earliest") \
        .option("failOnDataLoss", "false") \
        .load()
        
    raw_df = kafka_stream_df.selectExpr("CAST(value AS STRING) as json_payload")

    # 3. هيكل الـ JSON الكامل والمطابق لبيانات المحاكي
    data_schema = StructType([
        StructField("timestamp", StringType(), True),
        StructField("house_type", StringType(), True),
        StructField("currency", StringType(), True),
        StructField("zone", StringType(), True),
        StructField("device_id", StringType(), True),
        StructField("device_type", StringType(), True),
        StructField("is_room_occupied", BooleanType(), True), 
        StructField("power_consumption_watts", DoubleType(), True),
        StructField("status", StringType(), True)
    ])

    # 4. طبقة فك التشفير والتنظيف المتقدمة للبيانات واضطراب المستشعرات
    parsed_df = raw_df.withColumn("data", from_json(col("json_payload"), data_schema)) \
                     .select("data.*")

    final_processed_df = parsed_df.filter(
        (col("power_consumption_watts").isNotNull()) &
        (col("power_consumption_watts") >= 0.0) &
        (col("power_consumption_watts") <= 3500.0) &
        ~((col("device_type") == "Lighting") & (col("power_consumption_watts") > 100.0)) 
    )

    # 5. تحديد مسارات التخزين والـ Checkpoint لحفظ الإحداثيات (Parquet DWH)
    parquet_path = "/home/jovyan/work/storage/historical_parquet"
    checkpoint_path = "/home/jovyan/work/storage/checkp_parquet_increment_auto"

    # 🧹 تنظيف الـ Checkpoint القديم فوراً عند بدء التشغيل لتسريع العملية وتفادي البطء
    if os.path.exists(checkpoint_path):
        try:
            print("🧹 جاري تنظيف مجلد الـ Checkpoint القديم لتسريع التشغيل ومنع تضخم الملفات...")
            shutil.rmtree(checkpoint_path)
            print("✅ تم تصفير الـ Checkpoint بنجاح.")
        except Exception as e:
            print(f"⚠️ تنبيه: لم نتمكن من حذف مجلد الـ Checkpoint، قد يكون قيد الاستخدام: {e}")

    print("💾 جاري حقن البيانات بنمط تزايدي ذكي (Append) وتحديث الـ Checkpoint...")
    
    # ميزة availableNow=True تجعل سبارك يعمل كـ Batch (يعالج المتاح حالياً بكافكا ويتوقف تلقائياً)
    query = final_processed_df.writeStream \
        .format("parquet") \
        .outputMode("append") \
        .partitionBy("zone") \
        .option("path", parquet_path) \
        .option("checkpointLocation", checkpoint_path) \
        .trigger(availableNow=True) \
        .start()
        
    query.awaitTermination()
    print("✅ [نجاح ساحق] تم جلب البيانات الجديدة وحفظها في أرشيف الباركيه من حيث توقفنا سابقاً!")

except Exception as e:
    print(f"❌ فشل في محرك التنظيف التزايدي: {e}")

finally:
    spark.stop()

📡 جاري الاتصال بكافكا وسحب الدفعة التزايدية الجديدة...
🧹 جاري تنظيف مجلد الـ Checkpoint القديم لتسريع التشغيل ومنع تضخم الملفات...
✅ تم تصفير الـ Checkpoint بنجاح.
💾 جاري حقن البيانات بنمط تزايدي ذكي (Append) وتحديث الـ Checkpoint...
✅ [نجاح ساحق] تم جلب البيانات الجديدة وحفظها في أرشيف الباركيه من حيث توقفنا سابقاً!


import os
import sys

# 1. إعداد مسارات المكتبات لتعمل أوفلاين
offline_packages_path = "/home/jovyan/work/storage/packages"
if offline_packages_path not in sys.path:
    sys.path.insert(0, offline_packages_path)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType

internal_ivy_path = "/home/jovyan/.ivy2/jars/*"

# 2. بناء جلسة سبارك
spark = SparkSession.builder \
    .appName("Kafka_Preview_Latest_Data") \
    .config("spark.jars", internal_ivy_path) \
    .getOrCreate()

try:
    print("📡 جاري قراءة البيانات الحالية من كافكا لمعاينة آخر السجلات...")
    
    # قراءة كافكا بنظام الـ read (وليس readStream) لقراءة المتاح حالياً فقط والتوقف
    kafka_df = spark.read \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "smarthome-kafka:29092") \
        .option("subscribe", "energy_events") \
        .option("startingOffsets", "earliest") \
        .load()
        
    # تحويل القيمة القادمة من بايتس إلى نص JSON
    raw_df = kafka_df.selectExpr("CAST(value AS STRING) as json_payload", "timestamp as kafka_received_at")

    # 3. الهيكل الشامل لفك التشفير
    data_schema = StructType([
        StructField("timestamp", StringType(), True),
        StructField("house_type", StringType(), True),
        StructField("currency", StringType(), True),
        StructField("zone", StringType(), True),
        StructField("device_id", StringType(), True),
        StructField("device_type", StringType(), True),
        StructField("is_room_occupied", BooleanType(), True), 
        StructField("power_consumption_watts", DoubleType(), True),
        StructField("status", StringType(), True)
    ])

    # 4. فك تشفير الـ JSON واستخراج الأعمدة
    parsed_df = raw_df.withColumn("data", from_json(col("json_payload"), data_schema)) \
                     .select("data.*", "kafka_received_at")

    # 5. جلب آخر 5 سجلات بناءً على وقت وصولها لكافكا وعرضها
    print("\n📊 [معاينة حية] إليك آخر 5 سجلات متوفرة في الطابور:")
    
    # الترتيب التنازلي لإظهار الأحدث أولاً، ثم أخذ 5 فقط
    latest_5_df = parsed_df.orderBy(col("kafka_received_at").desc()).limit(5)
    
    # عرض النتيجة بشكل جدول منسق في الكونسول (مع منع قص النصوص الطويلة)
    latest_5_df.show(truncate=False)

except Exception as e:
    print(f"❌ فشل في جلب البيانات من كافكا: {e}")

finally:
    spark.stop()